# Email Triage RL — GRPO Training with Unsloth

**OpenEnv Hackathon 2026 — Team Ctrl-Alt-Defeat**

This notebook trains a small LLM to triage emails using **GRPO** (Group Relative Policy Optimisation) via TRL + Unsloth.

### What we train
- **Task**: Classify, prioritise, route, and respond to emails
- **Environment**: [`email_triage_env`](https://huggingface.co/spaces/your-org/email-triage-env)
- **Reward**: 4 independent verifier functions (classification, priority, routing, format)
- **Curriculum**: easy → medium → hard

### Runtime
- Google Colab T4 GPU (~45 min for full curriculum)
- Or A100 (~15 min)

## Step 1: Install dependencies

In [ ]:
%%capture
# Install Unsloth for efficient fine-tuning
!pip install unsloth
# Install TRL for GRPO
!pip install trl>=0.12.0 datasets accelerate peft bitsandbytes wandb
# Install environment
!pip install openenv>=0.1.13
# Clone the environment repo
!git clone https://huggingface.co/spaces/your-org/email-triage-env /content/email-triage-env 2>/dev/null || echo 'Already cloned'
import sys; sys.path.insert(0, '/content/email-triage-env')

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## Step 2: Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Small model, fits on T4
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Larger, needs A100
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,          # 4-bit quantisation for T4
    fast_inference=False,       # True for vLLM inference (optional)
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f'✅ Model loaded: {MODEL_NAME} (4-bit + LoRA r={LORA_RANK})')

## Step 3: Build training dataset from the environment

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = """You are an expert email triage agent. For each email, respond with a single valid JSON object:
{"email_id": "<id>", "category": "<spam|billing|technical|general|urgent>",
 "priority": <1-5>, "department": "<engineering|billing|support|management>",
 "response_draft": "<reply or null>", "escalate": <true|false>}
Respond with ONLY the JSON. No other text."""

def format_email_prompt(email, task_description):
    lines = [
        f"TASK: {task_description[:200]}", "",
        f"Email ID: {email.get('email_id', '')}",
        f"From: {email.get('sender_name', '')} <{email.get('sender', '')}>",
        f"Subject: {email.get('subject', '')}",
        f"Time: {email.get('timestamp', '')}",
        f"Has Attachment: {email.get('has_attachment', False)}",
        f"Is Reply: {email.get('is_reply', False)}",
    ]
    if email.get("thread_id"):
        lines.append(f"Thread ID: {email['thread_id']}")
    lines.extend(["", "Body:", email.get("body", "(empty)")])
    return "\n".join(lines)

def build_dataset(task_ids, seed=42):
    from server.email_generator import generate_emails
    from server.tasks import get_task
    examples = []
    for task_id in task_ids:
        task = get_task(task_id)
        emails, ground_truths = generate_emails(task_id, seed)
        truth_map = {gt.email_id: gt for gt in ground_truths}
        for email in emails:
            prompt = format_email_prompt(email.model_dump(), task.description)
            gt = truth_map[email.email_id]
            examples.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                "email_id": email.email_id,
                "task_id": task_id,
                "gt_category": gt.category,
                "gt_priority": gt.priority,
                "gt_department": gt.department,
                "gt_requires_response": gt.requires_response,
                "gt_expected_keywords": gt.expected_keywords,
                "requires_priority": task.requires_priority,
                "requires_routing": task.requires_routing,
                "requires_response": task.requires_response,
            })
    return examples

# Build curriculum dataset (all tasks)
raw_data = build_dataset(["easy", "medium", "hard"], seed=42)
dataset = Dataset.from_list(raw_data)
print(f'Dataset: {len(dataset)} examples')
print(f'Tasks: {set(dataset["task_id"])}')
print(f'\nSample prompt (first 500 chars):')
print(dataset[0]["prompt"][1]["content"][:500])

## Step 4: Define independent reward functions

In [ ]:
def _parse_action(completion):
    text = completion.strip()
    if "```" in text:
        lines = [l for l in text.split("\n") if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()
    try:
        a = json.loads(text)
        if isinstance(a, dict) and "email_id" in a:
            return a
    except json.JSONDecodeError:
        pass
    for i, ch in enumerate(text):
        if ch == "{":
            depth = 0
            for j in range(i, len(text)):
                if text[j] == "{": depth += 1
                elif text[j] == "}":
                    depth -= 1
                    if depth == 0:
                        try:
                            a = json.loads(text[i:j+1])
                            if isinstance(a, dict) and "email_id" in a:
                                return a
                        except: pass
                        break
    return None

VALID_CATS = {"spam", "billing", "technical", "general", "urgent"}
VALID_DEPTS = {"engineering", "billing", "support", "management"}

def reward_classification(completions, prompts=None, **kwargs):
    rewards = []
    gt_cats = kwargs.get("gt_category", ["general"] * len(completions))
    for comp, gt_cat in zip(completions, gt_cats):
        a = _parse_action(comp)
        if a is None: rewards.append(-0.2); continue
        cat = (a.get("category") or "").lower().strip()
        if cat == gt_cat: rewards.append(1.0)
        elif cat in VALID_CATS:
            close = {frozenset({"urgent","technical"}), frozenset({"billing","general"})}
            rewards.append(0.3 if frozenset({cat, gt_cat}) in close else -0.2)
        else: rewards.append(-0.5)
    return rewards

def reward_priority(completions, prompts=None, **kwargs):
    rewards = []
    gt_pris = kwargs.get("gt_priority", [3] * len(completions))
    requires = kwargs.get("requires_priority", [True] * len(completions))
    for comp, gt_pri, req in zip(completions, gt_pris, requires):
        if not req: rewards.append(0.0); continue
        a = _parse_action(comp)
        if a is None: rewards.append(-0.2); continue
        try:
            diff = abs(int(a.get("priority", 0)) - gt_pri)
            rewards.append(1.0 if diff == 0 else (0.4 if diff == 1 else -0.2))
        except: rewards.append(-0.2)
    return rewards

def reward_routing(completions, prompts=None, **kwargs):
    rewards = []
    gt_depts = kwargs.get("gt_department", ["support"] * len(completions))
    requires = kwargs.get("requires_routing", [True] * len(completions))
    for comp, gt_dept, req in zip(completions, gt_depts, requires):
        if not req: rewards.append(0.0); continue
        a = _parse_action(comp)
        if a is None: rewards.append(-0.2); continue
        dept = (a.get("department") or "").lower().strip()
        rewards.append(1.0 if dept == gt_dept else (-0.2 if dept in VALID_DEPTS else -0.5))
    return rewards

def reward_format(completions, prompts=None, **kwargs):
    rewards = []
    expected_ids = kwargs.get("email_id", [""] * len(completions))
    for comp, exp_id in zip(completions, expected_ids):
        a = _parse_action(comp)
        if a is None: rewards.append(-1.0); continue
        has_id = bool(a.get("email_id"))
        has_cat = bool(a.get("category"))
        id_ok = a.get("email_id") == exp_id
        rewards.append(0.5 if has_id and has_cat and id_ok else (0.2 if has_id and has_cat else -0.3))
    return rewards

print("✅ Reward functions defined")
# Quick sanity check
test_comp = '{"email_id": "e001", "category": "spam", "priority": 5, "department": "support", "escalate": false}'
print(f"Test classification reward: {reward_classification([test_comp], gt_category=['spam'])}")
print(f"Test format reward: {reward_format([test_comp], email_id=['e001'])}")

## Step 5: Run GRPO training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Training config — adjust for your GPU
TASK_ID = "easy"          # Start with easy; change to "curriculum" for full training
MAX_STEPS = 100           # Increase to 300+ for better results
NUM_GENERATIONS = 4       # G in GRPO — completions per prompt
BATCH_SIZE = 1            # T4: use 1; A100: use 2-4

# Filter dataset to single task (or use full curriculum)
task_dataset = dataset.filter(lambda x: x["task_id"] == TASK_ID)
print(f"Training on {TASK_ID}: {len(task_dataset)} examples")

grpo_config = GRPOConfig(
    output_dir=f"/content/checkpoints/email-triage-{TASK_ID}",
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=NUM_GENERATIONS,
    max_new_tokens=256,
    temperature=0.9,
    logging_steps=10,
    save_steps=50,
    seed=42,
    report_to="none",    # Change to "wandb" for W&B tracking
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_classification, reward_priority, reward_routing, reward_format],
    args=grpo_config,
    train_dataset=task_dataset,
    processing_class=tokenizer,
)

print(f"🚀 Starting GRPO training ({MAX_STEPS} steps, {NUM_GENERATIONS} generations/prompt)...")
train_result = trainer.train()
print(f"✅ Training complete! Final loss: {train_result.training_loss:.4f}")

## Step 6: Evaluate before vs. after training

In [ ]:
from server.environment import EmailTriageEnvironment

def run_eval(model, tokenizer, task_id, seed=99):
    """Run trained model against environment, return final score."""
    env = EmailTriageEnvironment()
    obs = env.reset(seed=seed, task_id=task_id)
    obs_dict = obs.model_dump()
    step_rewards = []

    while not obs_dict.get("done", False) and obs_dict.get("emails"):
        email = obs_dict["emails"][0]
        prompt = format_email_prompt(email, obs_dict.get("task_description", ""))
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            outputs = model.generate(inputs, max_new_tokens=256, do_sample=False)
        response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
        action = _parse_action(response)
        if action is None:
            action = {"email_id": email["email_id"], "category": "general",
                      "priority": 3, "department": "support"}
        obs = env.step(action)
        obs_dict = obs.model_dump()
        step_rewards.append(obs_dict.get("step_reward", 0.0))

    grading = obs_dict.get("metadata", {}).get("grading", {})
    return grading.get("final_score", 0.0), grading.get("dimension_scores", {}), step_rewards

print("Evaluating trained model...")
score, dims, rewards = run_eval(model, tokenizer, TASK_ID)
print(f"\nTask: {TASK_ID}")
print(f"Final score: {score:.4f}")
print(f"Dimensions: {json.dumps(dims, indent=2)}")
print(f"Step rewards: {[round(r, 2) for r in rewards]}")

## Step 7: Plot reward curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Extract training logs
log_history = trainer.state.log_history
train_steps = [l["step"] for l in log_history if "loss" in l]
train_loss = [l["loss"] for l in log_history if "loss" in l]
train_reward = [l.get("reward", None) for l in log_history if "loss" in l]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(train_steps, train_loss, 'b-', linewidth=1.5, label='Training Loss')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Loss')
axes[0].set_title(f'GRPO Training Loss — {TASK_ID.upper()}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Step rewards from evaluation
axes[1].bar(range(1, len(rewards)+1), rewards,
            color=['green' if r > 0 else 'red' for r in rewards], alpha=0.7)
axes[1].axhline(y=0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Email')
axes[1].set_ylabel('Step Reward')
axes[1].set_title(f'Per-Email Rewards — {TASK_ID.upper()} (score={score:.3f})')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'/content/training_results_{TASK_ID}.png', bbox_inches='tight')
plt.show()
print(f'Plot saved to: /content/training_results_{TASK_ID}.png')

## Step 8: Save the adapter (do NOT merge 4-bit weights)

In [ ]:
# Save LoRA adapter (safe — do NOT merge 4-bit model into 16-bit then save)
SAVE_PATH = f"/content/email-triage-adapter-{TASK_ID}"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Adapter saved to: {SAVE_PATH}")

# To push to HF Hub:
# model.push_to_hub("your-org/email-triage-grpo")
# tokenizer.push_to_hub("your-org/email-triage-grpo")

## Curriculum training (easy → medium → hard)

Re-run Step 5 with `TASK_ID = "medium"` after saving the easy checkpoint, then `TASK_ID = "hard"`.
Load the previous adapter before each stage:

```python
from peft import PeftModel
model = PeftModel.from_pretrained(model, "/content/email-triage-adapter-easy")
```

The curriculum schedule in `GET /curriculum` recommends:
- Train **easy** until avg eval score > 0.60
- Train **medium** until avg eval score > 0.50  
- Train **hard** for the final stage